# 目标检测

## 介绍

图像识别为整个图像分配单个标签，语义分割标记每个像素，但两者都无法区分单个对象。目标检测通过生成边界框来填补这一空白，每个边界框都有类别标签和置信度分数，用于定位和识别图像中的单个对象。

本教程涵盖地理空间图像目标检测的基础。您将探索主要架构系列，使用`geoai`包在[NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip)基准数据集上训练Faster R-CNN v2检测器，使用COCO风格指标评估结果，在新图像上运行推理，并将训练好的模型发布到Hugging Face Hub。

## 学习目标

在本教程结束时，您将能够：

- 解释目标检测与图像分类和语义分割的区别
- 描述边界框、置信度分数、非极大值抑制和锚框
- 比较两阶段、单阶段、基于Transformer和零样本检测架构
- 准备COCO标注格式的检测数据集
- 使用`geoai`包训练多类别目标检测模型
- 使用COCO风格的平均精度均值（mAP）评估检测性能
- 在新图像上运行推理并可视化检测结果
- 将训练好的模型发布到Hugging Face Hub并从托管模型运行推理

## 理解目标检测

### 分类与检测

图像分类为整个图像分配单个标签，但不说明特定对象在哪里或有多少个。目标检测生成可变数量的边界框，每个边界框都有类别标签和置信度分数，支持计数、定位和映射单个特征。

语义分割提供像素级标签但不分离单个对象。目标检测提供对象级定位但不描绘精确边界。实例分割结合了这两种能力。

### 关键概念

**边界框**是检测模型的基本输出，由指定检测对象周围矩形的四个坐标定义。在地理空间应用中，这些像素空间坐标通过图像的仿射变换转换为地理坐标。

**置信度分数**伴随每个边界框，范围从0到1，表示模型对预测的确定性。高阈值减少假阳性但可能遗漏对象，而低阈值捕获更多对象但引入更多假检测。

**交并比（IoU）**衡量预测边界框与真实框的重叠程度，计算为重叠面积除以并集面积。它在训练和评估期间都使用。

**非极大值抑制（NMS）**是一种后处理步骤，通过只为每个对象保留最高置信度的框来去除冗余的重叠检测。这对于在密集场景中生成清晰输出至关重要。

**锚框**是不同尺度和宽高比的预定义边界框模板，作为预测的起点。模型学习相对于这些锚点的小偏移，而不是从头预测坐标。

## 检测架构

### 两阶段检测器

两阶段检测器将检测分为候选区域生成和分类。**Faster R-CNN**首先使用区域候选网络（RPN）生成候选区域，然后通过专用头对每个候选区域进行分类和细化。

两阶段检测器实现高精度并很好地处理多尺度对象，使其适用于地理空间图像。然而，其顺序性质使其比单阶段替代方案慢。

### 单阶段检测器

单阶段检测器在单次遍历中直接从特征图预测边界框和类别标签，使其明显更快。

- **YOLO**（You Only Look Once）将检测构建为密集预测问题，并演变为有效平衡速度和准确性的模型系列。
- **SSD**（Single Shot MultiBox Detector）从不同分辨率的多个特征图进行预测，检测不同尺度的对象。
- **RetinaNet**使用焦点损失函数通过将训练集中在困难样本上来解决类别不平衡问题。在`geoai`中通过`retinanet_resnet50_fpn_v2`支持。
- **FCOS**是一种无锚点检测器，直接在每个空间位置预测边界框。在`geoai`中通过`fcos_resnet50_fpn`支持。

### 基于Transformer的检测器

[DETR](https://huggingface.co/docs/transformers/model_doc/detr)（DEtection TRansformer）使用Transformer编码器-解码器架构将目标检测构建为集合预测问题，消除了对锚点和NMS的需求。其主要优势是简单性和全局上下文推理，尽管收敛速度较慢，可能在处理非常小的对象时遇到困难。

### 零样本检测

零样本检测模型识别由文本提示描述的对象，无需特定任务的训练数据。[OWL-ViT](https://huggingface.co/docs/transformers/en/model_doc/owlvit)将视觉Transformer与文本编码器相结合，支持通过"太阳能电池板"或"游泳池"等自然语言描述进行检测。

[Grounding DINO](https://huggingface.co/docs/transformers/en/model_doc/grounding-dino)通过将基于DINO的检测与接地语言理解相结合来扩展这一范式，实现强大的零样本性能。

### 选择架构

架构的选择取决于应用需求：

- **Faster R-CNN**是当准确性优先且推理速度不太关键时的强大默认选择。它很好地处理多尺度对象，是`geoai`包检测管道中的默认架构。
- **YOLO**在处理速度很重要时是首选，例如扫描大型卫星图像档案或支持近实时监控应用。
- **DETR及其变体**适用于需要全局上下文推理的场景以及您希望避免调整锚框配置的情况。
- **零样本检测器**（OWL-ViT、Grounding DINO）非常适合探索性分析、快速原型开发或无法获得标注训练数据的应用。

`geoai`包通过`model_name`参数支持多种检测架构：

| Model Name                          | Type         | Notes                                  |
| ----------------------------------- | ------------ | -------------------------------------- |
| `fasterrcnn_resnet50_fpn_v2`        | 两阶段       | 默认，良好的准确性/速度权衡            |
| `fasterrcnn_mobilenet_v3_large_fpn` | 两阶段       | 最快的两阶段选项                       |
| `retinanet_resnet50_fpn_v2`         | 单阶段       | 快速，很好地处理类别不平衡             |
| `fcos_resnet50_fpn`                 | Single-stage | Anchor-free                            |
| `maskrcnn_resnet50_fpn`             | 两阶段       | 还生成实例掩码（最慢）                 |

实际上，许多项目从预训练或零样本模型开始评估可行性，然后转向特定任务的检测器进行生产。将检测器的优势与目标对象的规模相匹配通常比选择最新架构更重要。

## 准备检测数据集

### 标注格式

**COCO格式**将标注存储在单个JSON文件中，包含图像元数据、类别定义和边界框（以绝对像素坐标表示为(x, y, width, height)）。这是`geoai`包使用的格式。

**YOLO格式**每个图像使用一个文本文件，每行描述一个对象，表示为归一化坐标中的`class_id center_x center_y width height`。

### NWPU-VHR-10数据集

[NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip)数据集是超高分辨率遥感图像目标检测的广泛使用的基准数据集。它包含800张图像（650张正样本，150张仅背景），涵盖10个类别：飞机、船舶、储罐、棒球场、网球场、篮球场、田径场、港口、桥梁和车辆。

`geoai`包提供`prepare_nwpu_vhr10`函数，自动将标注图像分割为训练集和验证集，并生成COCO格式标注。

## 评估检测结果

### 平均精度均值（mAP）

如果检测结果与真实框的IoU超过阈值且类别正确，则为**真阳性**；否则为**假阳性**。未匹配的真实框是**假阴性**。

单个类别的**平均精度（AP）**是精确率-召回率曲线下的面积。**平均精度均值（mAP）**是所有类别的AP平均值。

### 精确率-召回率曲线

精确率-召回率曲线绘制置信度阈值变化时的精确率与召回率关系。强大的检测器即使在高召回率下也能保持高精度。检查每类曲线可以揭示模型在哪些对象类型上存在困难。

### IoU阈值

- **mAP@0.5**要求至少50%的重叠，在近似定位足够的地理空间应用中很常见。
- **mAP@0.75**要求75%的重叠，奖励更精确的定位。
- **mAP@0.5:0.95**是0.5到0.95阈值（步长0.05）的平均值，是主要的COCO基准指标。

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 导入库

`geoai`包提供完整目标检测管道的函数，包括数据集准备、训练、评估、推理和模型共享。

In [ ]:
import os
import json

import geoai

## 下载NWPU-VHR-10数据集

NWPU-VHR-10数据集作为托管在Source Cooperative上的zip存档提供。`download_file`工具自动下载并解压它。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip"
data_dir = geoai.download_file(url)

In [ ]:
print(f"Dataset directory: {data_dir}")
print(f"Contents: {os.listdir(data_dir)}")

## 探索数据集

数据集包含10个对象类别以及索引0处的背景类别。

In [ ]:
print(f"\nNWPU-VHR-10 Classes:")
for i, name in enumerate(geoai.NWPU_VHR10_CLASSES):
    print(f"  {i}: {name}")

```text
NWPU-VHR-10类别：
  0: background
  1: airplane
  2: ship
  3: storage_tank
  4: baseball_diamond
  5: tennis_court
  6: basketball_court
  7: ground_track_field
  8: harbor
  9: bridge
  10: vehicle
```

## 准备数据集

`prepare_nwpu_vhr10`函数将正样本图像分割为训练集和验证集，并为每个分割生成COCO格式的标注文件。

In [ ]:
splits = geoai.prepare_nwpu_vhr10(data_dir, val_split=0.2, seed=42)

In [ ]:
print(f"Images directory: {splits['images_dir']}")
print(f"Number of classes: {splits['num_classes']}")
print(f"Class names: {splits['class_names']}")
print(f"Training images: {len(splits['train_image_ids'])}")
print(f"Validation images: {len(splits['val_image_ids'])}")

```text
图像目录：./NWPU-VHR-10/positive_image_set
类别数量：11
类别名称：['background', 'airplane', 'ship', 'storage_tank', 'baseball_diamond', 'tennis_court', 'basketball_court', 'ground_track_field', 'harbor', 'bridge', 'vehicle']
训练图像：509
验证图像：128
```

## 可视化示例标注

在训练前可视化带有真实边界框的示例图像以验证标注。

In [ ]:
geoai.visualize_coco_annotations(
    annotations_path=splits["annotations_path"],
    images_dir=splits["images_dir"],
    num_samples=6,
    random=True,
    seed=1,
    cols=3,
    figsize=(12, 6),
)

## 训练多类别检测模型

`train_multiclass_detector`函数处理完整的训练管道：加载数据、构建模型、运行训练循环并保存最佳检查点。关键参数包括模型架构、类别名称、epoch数量、批量大小和学习率。

In [ ]:
output_dir = "nwpu_output"

model_path = geoai.train_multiclass_detector(
    images_dir=splits["images_dir"],
    annotations_path=splits["train_annotations"],
    output_dir=output_dir,
    model_name="fasterrcnn_resnet50_fpn_v2",
    class_names=splits["class_names"],
    num_channels=3,
    batch_size=4,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.1,
    seed=42,
    pretrained=True,
    verbose=True,
)

## 绘制训练指标

绘制训练损失、验证IoU和学习率调度随epoch的变化，以评估模型收敛情况。

In [ ]:
geoai.plot_detection_training_history(
    history_path=os.path.join(output_dir, "training_history.pth"),
)

## 使用COCO指标评估

`evaluate_multiclass_detector`函数在验证集上计算COCO风格的mAP指标，包括每类AP分数。

In [ ]:
metrics = geoai.evaluate_multiclass_detector(
    model_path=model_path,
    images_dir=splits["images_dir"],
    annotations_path=splits["val_annotations"],
    num_classes=splits["num_classes"],
    class_names=splits["class_names"][1:],  # Exclude background
    batch_size=4,
)

```text
评估结果：
  mAP@0.5:        0.7312
  mAP@0.75:       0.4936
  mAP@[0.5:0.95]: 0.4428

  每类AP@0.5：
    AP@0.5/airplane: 0.7106
    AP@0.5/baseball_diamond: 0.7885
    AP@0.5/basketball_court: 0.8957
    AP@0.5/bridge: 0.9052
    AP@0.5/ground_track_field: 0.7081
    AP@0.5/harbor: 0.5322
    AP@0.5/ship: 0.6349
    AP@0.5/storage_tank: 0.5624
    AP@0.5/tennis_court: 0.8967
    AP@0.5/vehicle: 0.6781
```

像篮球场和网球场这样大而独特的对象比像车辆和储罐这样较小或更模糊的对象获得更高的AP。

## 在示例图像上运行推理

`multiclass_detection`函数处理切片、模型推理、跨瓦片边界的NMS和结果组装。

In [ ]:
# Load validation data to pick a test image
with open(splits["val_annotations"], "r") as f:
    val_data = json.load(f)

test_img_info = val_data["images"][0]
test_img_path = os.path.join(splits["images_dir"], test_img_info["file_name"])
print(f"Test image: {test_img_path}")

In [ ]:
output_raster = "nwpu_detection_output.tif"

result_path, inference_time, detections = geoai.multiclass_detection(
    input_path=test_img_path,
    output_path=output_raster,
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

print(f"\nInference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

```text
NMS前收集了27个检测
NMS后：7个检测
多类别检测在0.14秒内完成
最终检测：7
  港口：6个检测
  桥梁：1个检测
多类别检测结果保存到nwpu_detection_output.tif

推理时间：0.14s
总检测数：7
```

## 可视化检测

`visualize_multiclass_detections`函数将检测到的边界框叠加在原始图像上，按类别着色并用置信度分数标注。

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=test_img_path,
    detections=detections,
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## 批量推理多张图像

`batch_multiclass_detection`函数在多张图像上运行推理并生成可视化网格。

In [ ]:
val_image_paths = [
    os.path.join(splits["images_dir"], img["file_name"])
    for img in val_data["images"][:4]
]

results = geoai.batch_multiclass_detection(
    image_paths=val_image_paths,
    output_dir="nwpu_batch_output",
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    num_channels=3,
    figsize=(16, 12),
)

预测并不完美。您可能会注意到结果中存在假阳性和假阴性。

## 发布和重用模型

### 推送到Hugging Face Hub

通过Hugging Face Hub共享训练好的模型，让协作者可以在没有原始训练数据或计算资源的情况下运行推理。

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
url = geoai.push_detector_to_hub(
    model_path=model_path,
    repo_id="your-username/nwpu-vhr10-fasterrcnn",
    model_name="fasterrcnn_resnet50_fpn_v2",
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
)

### 从Hub运行推理

`predict_detector_from_hub`函数从Hub下载模型并自动运行推理，不需要本地检查点或类别名称列表。

In [ ]:
sample_img_path = os.path.join(splits["images_dir"], "608.jpg")

result_path, inference_time, detections = geoai.predict_detector_from_hub(
    input_path=sample_img_path,
    output_path="hub_detection.tif",
    repo_id="giswqs/nwpu-vhr10-fasterrcnn",
    confidence_threshold=0.5,
)

print(f"Inference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

# Clean up
if os.path.exists("hub_detection.tif"):
    os.remove("hub_detection.tif")

```text
NMS前收集了34个检测
NMS后：8个检测
多类别检测在0.13秒内完成
最终检测：8
  棒球场：1个检测
  网球场：4个检测
  篮球场：3个检测
多类别检测结果保存到hub_detection.tif
Inference time: 0.13s
Total detections: 8
```

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=sample_img_path,
    detections=detections,
    class_names=geoai.NWPU_VHR10_CLASSES,
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## 关键要点

1. 目标检测使用边界框定位和分类单个对象，填补了图像分类和像素级分割之间的空白。

2. 边界框、IoU、NMS和锚框是检测模型生成、细化和过滤预测的基础概念。

3. 两阶段检测器（Faster R-CNN）优先考虑准确性，单阶段检测器（YOLO、RetinaNet、FCOS）优先考虑速度，基于Transformer的检测器（DETR）简化管道，零样本检测器（OWL-ViT、Grounding DINO）消除了对特定任务训练数据的需求。

4. NWPU-VHR-10数据集为遥感多类别目标检测提供标准的10类别基准。

5. `geoai`包简化了从数据集准备到训练、评估、推理和模型共享的完整检测工作流程。

6. 多IoU阈值的COCO风格mAP指标提供全面评估，每类AP揭示优势和劣势。

7. 置信度阈值调整平衡精确率和召回率，最佳阈值取决于假阳性与假阴性的特定应用成本。

8. Publishing models to Hugging Face Hub enables sharing and reuse without requiring local training infrastructure.